# Experimento: Abliteración Quirúrgica vs. Clásica en `LiquidAI/LFM3-1.7B` (OR-Bench, Colab L4)

Prueba el caso de uso de **seguridad anti-backdoor, velocidad y cero lobotomía** de `.moipatch` sobre prompts **100% benignos** de falso rechazo (`bench-llm/or-bench` `or-bench-hard-1k`) + controles sanos:
1. **Piloto + STOP Gate (`N_PILOT=80`):** si `LFM3-1.7B` base casi no rechaza en falso, se detiene enseguida sin quemar créditos.
2. **Baseline Clásico (fuerza bruta):** ablitera la dirección de rechazo en **las 48 matrices residuales** (`ALL layers`, permanente).
3. **WSI Quirúrgico (`Unión Pasivo + Causal` + Juez Forward):** `capture_causal_trace` + `diagnose_causal_traces` + `screen_patch_candidates` eligen solo **1–2 módulos**, ajustan un **Router MoI** en `Half-A` y lo aplican en 1 línea con `patch.enable_routed(model)`.
4. **Auditoría Anti-Backdoor:** reporta peso exacto en KB, fracción de parámetros modificados, firma SHA-256 y proyección al vocabulario (`logit-lens`) sobre lo que escribe el parche.

Guarda el JSON final + el archivo `.moipatch` atómicamente en `MyDrive/wsi_runs/FINALES/`.

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
RUN_NAME = "surgical_refusal_lfm3_l4"
REPO = "/content/wsi"
BASE = "/content/base"
N_PILOT = 80
N_OR_BENCH = 160
N_CONTROL = 60
N_CAUSAL_TRACES = 10
MAX_NEW_TOK = 64

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers accelerate safetensors pandas pyarrow huggingface_hub hf_transfer
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(FINALES, exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| modelo en:", BASE)

In [ ]:
import os, subprocess, sys

subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

LOCAL_OUT = f"/content/{RUN_NAME}.json"
LOCAL_PATCH = "/content/lfm3_surgical_refusal.moipatch"
SCRIPT = f"{REPO}/scripts/experiment_surgical_refusal_lfm3.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}). Falta push en local.")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--n-pilot", str(N_PILOT),
    "--n-or-bench", str(N_OR_BENCH),
    "--n-control", str(N_CONTROL),
    "--n-causal-traces", str(N_CAUSAL_TRACES),
    "--max-new-tokens", str(MAX_NEW_TOK),
    "--patch-out", LOCAL_PATCH,
    "--out", LOCAL_OUT,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
import datetime, json, os, shutil, subprocess

res = json.load(open(LOCAL_OUT, encoding="utf-8"))
print("STATUS:", res["status"], "| PILOTO:", res.get("pilot"))

if res["status"] == "ok":
    print("\n=== COMPARATIVA EN HALF-B HELD-OUT (LFM3-1.7B) ===")
    for name, a in res["arms"].items():
        print(f"{name:26s} | over-refusal={a['or_bench_refusal_rate']:6.1%} | recov={str(a['recovered_of_base_refused']):>6} "
              f"| ctrl_exact={a['control_exact_match_rate']:6.1%} | healthy_kl={a['healthy_kl']:7.5f} | dNLL={a['delta_healthy_nll']:+7.5f}")
    au = res["audit"]
    print("\n=== AUDITORÍA ANTI-BACKDOOR ===")
    print(f"Tamaño .moipatch: {au['patch_size_kb']} KB | Módulos editados: {au['n_edited_modules']}/{res['architecture']['n_residual_writers']} "
          f"(vs Clásico: {au['classic_n_edited_modules']}) | SHA256 OK: {au['fingerprint_ok']}")
    for v in au["vocab_logit_lens"]:
        print(f"  {v['module']} (alpha={v['alpha']}, cols={v['cols']}):")
        print("    suprime :", [x["token"] for x in v["suppressed_tokens"][:6]])
        print("    promueve:", [x["token"] for x in v["promoted_tokens"][:6]])
    print("\nTiming (s):", res["timing_s"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "surgical_refusal_lfm3_colab.ipynb",
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2)
os.replace(dst + ".tmp", dst)
if os.path.exists(LOCAL_PATCH):
    shutil.copy2(LOCAL_PATCH, f"{FINALES}/lfm3_surgical_refusal.moipatch")
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar runtime solo si el JSON final quedó guardado y verificado en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")